In [4]:
import yaml
import requests
import json

# 1. Load your file
FILE_NAME = 'aircraft-speeds_lt_YES_1_a817d345.yaml'
with open(FILE_NAME, 'r') as f:
    data = yaml.safe_load(f)

# 2. Setup
# Replace with your actual OpenRouter API Key
API_KEY = "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"
MODEL = "google/gemini-2.0-flash-001"
URL = "https://openrouter.ai/api/v1/chat/completions"

def ask_openrouter(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Answer with ONLY 'YES' or 'NO'."},
            {"role": "user", "content": prompt}
        ]
    }
    try:
        response = requests.post(URL, headers=headers, data=json.dumps(payload))
        return response.json()['choices'][0]['message']['content'].strip().upper()
    except Exception as e:
        return f"ERROR"

# 3. Process the questions
questions = data['question_by_qid']
print(f"--- Starting Full Comparative Evaluation ---\n")

for qid, info in questions.items():
    x_name, x_val = info['x_name'], info['x_value']
    y_name, y_val = info['y_name'], info['y_value']

    # --- PART A: INTERNAL KNOWLEDGE (BLIND GUESS) ---
    # Original Order
    ans_guess_orig = ask_openrouter(f"Is the {x_name} slower than the {y_name}?")
    # Reversed Order
    ans_guess_rev = ask_openrouter(f"Is the {y_name} faster than the {x_name}?")

    # --- PART B: GROUNDED VERIFICATION (WITH DATA) ---
    facts = f"Fact 1: The {x_name} has a top speed of {x_val} km/h.\nFact 2: The {y_name} has a top speed of {y_val} km/h."
    
    # Original Order Grounded
    ans_ground_orig = ask_openrouter(f"{facts}\nQuestion: Based on these facts, is {x_name} slower than {y_name}?")
    # Reversed Order Grounded
    ans_ground_rev = ask_openrouter(f"{facts}\nQuestion: Based on these facts, is {y_name} faster than {x_name}?")

    # --- DATA INTEGRITY CHECK ---
    # Does the YAML math actually work? (X should be < Y for a 'YES' answer)
    yaml_math_correct = "CORRECT" if x_val < y_val else "INCORRECT"

    # --- OUTPUT RESULTS ---
    print(f"AIRCRAFT PAIR: {x_name} vs {y_name}")
    print(f"  [Data Check] YAML Logic: {yaml_math_correct} ({x_val} < {y_val})")
    print(f"  [Blind Guess] Is {x_name} slower? : {ans_guess_orig}")
    print(f"  [Blind Guess] Is {y_name} faster? : {ans_guess_rev}")
    print(f"  [Grounded]    Is {x_name} slower? : {ans_ground_orig}")
    print(f"  [Grounded]    Is {y_name} faster? : {ans_ground_rev}")
    
    # Highlighting potential issues
    if ans_guess_orig != ans_guess_rev:
        print("  ⚠️ ALERT: Model is inconsistent in its internal knowledge (order bias).")
    if ans_ground_orig != "YES" or ans_ground_rev != "YES":
        print("  ❌ FAIL: Model failed to follow the provided numerical facts.")
        
    print("-" * 65)

print("\nEvaluation Complete.")

--- Starting Full Comparative Evaluation ---

AIRCRAFT PAIR: Boeing 787 Dreamliner vs Boeing 747-400
  [Data Check] YAML Logic: CORRECT (954 < 988)
  [Blind Guess] Is Boeing 787 Dreamliner slower? : NO
  [Blind Guess] Is Boeing 747-400 faster? : YES
  [Grounded]    Is Boeing 787 Dreamliner slower? : YES
  [Grounded]    Is Boeing 747-400 faster? : YES
  ⚠️ ALERT: Model is inconsistent in its internal knowledge (order bias).
-----------------------------------------------------------------
AIRCRAFT PAIR: Concorde vs MiG-21
  [Data Check] YAML Logic: CORRECT (2179 < 2230)
  [Blind Guess] Is Concorde slower? : NO
  [Blind Guess] Is MiG-21 faster? : NO
  [Grounded]    Is Concorde slower? : YES
  [Grounded]    Is MiG-21 faster? : YES
-----------------------------------------------------------------
AIRCRAFT PAIR: Boeing 737-800 vs Boeing 787 Dreamliner
  [Data Check] YAML Logic: CORRECT (946 < 954)
  [Blind Guess] Is Boeing 737-800 slower? : YES
  [Blind Guess] Is Boeing 787 Dreamliner faste